In [1]:
from Bio import SeqIO
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:

def calculate_gc_profile(fasta_file, window_size=1000, step_size=1000):
    """
    Calculate GC content profile for sequences in a FASTA file using sliding windows.
    
    Parameters:
    -----------
    fasta_file : str
        Path to the input FASTA file
    window_size : int
        Size of the sliding window (default: 1000 bp)
    step_size : int
        Step size for sliding window (default: 1000 bp)
    
    Returns:
    --------
    dict
        Dictionary with chromosome names as keys and DataFrames containing position and GC content as values
    """
    gc_profiles = {}
    
    for record in SeqIO.parse(fasta_file, "fasta"):
        sequence = str(record.seq).upper()
        positions = []
        gc_contents = []
        
        # Calculate GC content for each window
        for i in range(0, len(sequence) - window_size + 1, step_size):
            window = sequence[i:i + window_size]
            gc_count = window.count('G') + window.count('C')
            gc_content = (gc_count / window_size) * 100
            
            positions.append(i)
            gc_contents.append(gc_content)
        
        # Store results in a DataFrame
        gc_profiles[record.id] = pd.DataFrame({
            'Position': positions,
            'GC_Content': gc_contents
        })
    
    return gc_profiles

def plot_gc_profile(gc_profiles, output_prefix="gc_profile"):
    """
    Plot GC content profiles for each chromosome.
    
    Parameters:
    -----------
    gc_profiles : dict
        Dictionary containing GC profile DataFrames for each chromosome
    output_prefix : str
        Prefix for output plot files
    """
    for chrom, data in gc_profiles.items():
        plt.figure(figsize=(15, 5))
        plt.plot(data['Position'], data['GC_Content'])
        plt.title(f'GC Content Profile - {chrom}')
        plt.xlabel('Position (bp)')
        plt.ylabel('GC Content (%)')
        plt.grid(True, alpha=0.3)
        plt.savefig(f'{output_prefix}_{chrom}.png')
        plt.close()



In [3]:
def main():
    # Example usage
    fasta_file = "../Reference/subset_lawson.fasta"  # Replace with your FASTA file path
    window_size = 50000 #1000
    step_size = 1000 #1000
    
    # Calculate GC profiles
    gc_profiles = calculate_gc_profile(fasta_file, window_size, step_size)
    
    # Generate summary statistics
    summary_stats = {}
    for chrom, data in gc_profiles.items():
        summary_stats[chrom] = {
            'Mean_GC': data['GC_Content'].mean(),
            'Std_GC': data['GC_Content'].std(),
            'Min_GC': data['GC_Content'].min(),
            'Max_GC': data['GC_Content'].max()
        }
    
    # Save summary statistics
    #pd.DataFrame(summary_stats).T.to_csv('gc_content_summary.csv')
    print(pd.DataFrame(summary_stats))
    
    # Plot profiles
    plot_gc_profile(gc_profiles)

if __name__ == "__main__":
    main()

              chr1      chr10      chr11      chr12      chr13      chr14  \
Mean_GC  36.348001  36.515197  36.289517  36.202944  36.399946  36.534907   
Std_GC    2.044480   1.921944   1.770796   1.882256   1.816396   1.965680   
Min_GC   29.886000  31.730000  30.724000  29.132000  31.524000  31.080000   
Max_GC   50.754000  61.598000  47.398000  49.948000  48.182000  58.424000   

             chr15      chr16      chr17      chr18  ...      chr23  \
Mean_GC  36.730086  36.492885  36.549690  36.488651  ...  36.669021   
Std_GC    1.984019   1.936747   2.124754   2.000366  ...   1.961789   
Min_GC   31.394000  29.394000  30.906000  28.840000  ...  29.890000   
Max_GC   49.356000  57.496000  62.282000  50.634000  ...  50.272000   

             chr24      chr25       chr3      chr4       chr5       chr6  \
Mean_GC  36.178890  36.512650  36.851387  36.56785  36.353187  36.364893   
Std_GC    1.771231   1.981284   2.158105   7.09355   1.824396   1.938684   
Min_GC   30.552000  30.022000 